# Münster water extremes: too much, then too little

Starter notebook for the Münster heavy-rain & drought tracker.

**What it does**
1. Pulls hourly precipitation for the DWD station **Münster/Osnabrück** via the free [Bright Sky API](https://brightsky.dev) (open DWD data), 2010 to today, cached locally.
2. Builds clean daily totals (days with too few hourly readings are marked missing, not zero).
3. Makes three charts for your first post:
   - the last 7 days of rain vs. the same week in previous years
   - this year's cumulative rainfall vs. the 2010+ range (both extremes in one picture)
   - summer rain totals and longest dry spell by year
4. Prints the headline numbers and a draft post.

Install: `pip install requests pandas matplotlib`

Data: Deutscher Wetterdienst (DWD), via Bright Sky. Credit "Quelle: Deutscher Wetterdienst" in posts.

In [ ]:
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ---- Config ---------------------------------------------------------------
STATION_ID = "01766"        # DWD Münster/Osnabrück (airport, Greven)
STATION_NAME = "Münster/Osnabrück"
TZ = "Europe/Berlin"
BASELINE_START = 2010       # Bright Sky's historical archive starts around here
DRY_DAY_MM = 1.0            # a day with < 1 mm counts as "dry" (common climatology threshold)
MIN_HOURS = 20              # need at least 20 of 24 hourly readings for a valid day
SUMMER_MONTHS = (6, 7, 8)   # June-August

NOW = pd.Timestamp.now(tz=TZ)
TODAY = NOW.normalize()
SUMMER_YEAR = TODAY.year if TODAY.month > max(SUMMER_MONTHS) else TODAY.year - 1

CACHE = Path("data/cache"); CACHE.mkdir(parents=True, exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)

# ---- Chart style (one consistent, readable look) --------------------------
BLUE, BLUE_LIGHT, ORANGE = "#2a78d6", "#b7d3f6", "#eb6834"
GRAY_BAR, INK, INK_2, MUTED, GRID, SURFACE = "#d6d5ce", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 11,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.titleweight": "bold", "axes.titlesize": 14, "axes.titlelocation": "left", "axes.titlecolor": INK,
})
SOURCE_NOTE = f"Data: Deutscher Wetterdienst, station {STATION_NAME}, via Bright Sky"

## 1. Fetch hourly data (cached per month)

In [ ]:
API = "https://api.brightsky.dev/weather"

def fetch_month(year: int, month: int) -> pd.DataFrame:
    """Hourly observations for one month. Past months are cached; the current month is always refetched."""
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)
    is_recent = end.tz_localize(TZ) > TODAY - pd.Timedelta(days=10)   # recent data can still get corrected
    cache_file = CACHE / f"{STATION_ID}_{year}-{month:02d}.csv"
    if cache_file.exists() and not is_recent:
        return pd.read_csv(cache_file)

    params = {"dwd_station_id": STATION_ID, "date": start.date().isoformat(),
              "last_date": end.date().isoformat(), "tz": TZ}
    for attempt in range(3):
        try:
            r = requests.get(API, params=params, timeout=60)
            r.raise_for_status()
            break
        except requests.RequestException as e:
            if attempt == 2:
                raise
            time.sleep(2 * (attempt + 1))
    payload = r.json()

    # Keep observations only - drop forecast (MOSMIX) rows that cover the future
    obs_types = {s["id"]: s.get("observation_type") for s in payload.get("sources", [])}
    df = pd.DataFrame(payload.get("weather", []))
    if df.empty:
        return df
    df["observation_type"] = df["source_id"].map(obs_types)
    df = df[df["observation_type"] != "forecast"][["timestamp", "precipitation", "observation_type"]]
    if not is_recent:
        df.to_csv(cache_file, index=False)
    time.sleep(0.2)  # be polite to a free API
    return df


months = pd.period_range(f"{BASELINE_START}-01", TODAY.strftime("%Y-%m"), freq="M")
frames = []
for i, p in enumerate(months, 1):
    frames.append(fetch_month(p.year, p.month))
    if i % 24 == 0 or i == len(months):
        print(f"fetched {i}/{len(months)} months")

hourly = pd.concat([f for f in frames if not f.empty], ignore_index=True)
hourly["timestamp"] = pd.to_datetime(hourly["timestamp"], utc=True).dt.tz_convert(TZ)
# Prefer the quality-checked 'historical' record when several sources cover the same hour
priority = {"historical": 0, "current": 1, "synop": 2}
hourly["prio"] = hourly["observation_type"].map(priority).fillna(9)
hourly = (hourly.sort_values(["timestamp", "prio"])
                .drop_duplicates("timestamp")
                .query("timestamp <= @NOW")
                .drop(columns="prio"))
print(hourly["observation_type"].value_counts())
hourly.tail()

## 2. Daily totals with honest gaps

In [ ]:
agg = hourly.set_index("timestamp")["precipitation"].resample("D").agg(["sum", "count"])
daily = agg["sum"].where(agg["count"] >= MIN_HOURS)       # too few readings -> NaN, not 0 mm
daily = daily[daily.index < TODAY]                          # today is incomplete
daily.index = daily.index.tz_localize(None)
daily.name = "precip_mm"

coverage = daily.groupby(daily.index.year).apply(lambda s: s.notna().mean()).rename("valid_day_share")
print("Share of valid days per year (check before trusting a year):")
print(coverage.round(3).to_string())
weak_years = coverage[coverage < 0.9].index.tolist()
if weak_years:
    print(f"\n⚠ Years with <90% valid days: {weak_years} - treat their numbers with care.")

## 3. Chart 1 - the last 7 days vs. the same week in other years

In [ ]:
week_end = daily.index.max()
week_start = week_end - pd.Timedelta(days=6)
week = daily[week_start:week_end]
week_total = week.sum()

same_week = {}
for y in range(BASELINE_START, week_end.year):
    shift = pd.DateOffset(years=week_end.year - y)
    s = daily[week_start - shift: week_end - shift]
    if s.notna().sum() >= 6:
        same_week[y] = s.sum()
same_week = pd.Series(same_week)
avg_week = same_week.mean()
rank = int((same_week > week_total).sum()) + 1   # 1 = wettest

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(week.index, week.values, width=0.8, color=BLUE, zorder=2)
for d, v in week.items():
    if v >= 1:
        ax.text(d, v, f"{v:.0f}", ha="center", va="bottom", fontsize=10, color=INK_2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a\n%d %b"))
ax.set_ylabel("Rain per day (mm)")
ax.set_title(f"Münster, last 7 days: {week_total:.0f} mm of rain", pad=26)
ax.text(0, 1.04, f"The same week averaged {avg_week:.0f} mm in {same_week.index.min()}–{same_week.index.max()}  "
        f"·  #{rank} wettest of {len(same_week) + 1} years",
        transform=ax.transAxes, color=INK_2, fontsize=10.5)
fig.text(0.01, 0.01, SOURCE_NOTE, color=MUTED, fontsize=8)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(FIG / "01_last_7_days.png")
plt.show()

## 4. Chart 2 - this year's cumulative rain vs. every year since 2010

The hero chart: a dry summer shows up as a flat line, a wet week as a jump.

In [ ]:
cum = daily.fillna(0).to_frame()
cum["year"], cum["doy"] = cum.index.year, cum.index.dayofyear
cum["cum"] = cum.groupby("year")["precip_mm"].cumsum()
table = cum.pivot_table(index="doy", columns="year", values="cum")
table = table.loc[table.index <= 365]   # day 366 exists only in leap years and distorts the band

this_year = TODAY.year
past = table.drop(columns=[c for c in [this_year] + weak_years if c in table.columns])
band_lo, band_hi, band_mean = past.min(axis=1), past.max(axis=1), past.mean(axis=1)
cur = table[this_year].dropna() if this_year in table.columns else pd.Series(dtype=float)
ref_date = pd.Timestamp("2001-01-01")  # non-leap reference year for month ticks
to_dates = lambda idx: ref_date + pd.to_timedelta(idx - 1, unit="D")

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(to_dates(past.index), band_lo, band_hi, color=BLUE_LIGHT, alpha=0.6, lw=0, label=f"Range {past.columns.min()}–{past.columns.max()}")
ax.plot(to_dates(past.index), band_mean, color=MUTED, lw=1.5, ls="--", label="Average")
ax.plot(to_dates(cur.index), cur.values, color=BLUE, lw=2.5, label=str(this_year), zorder=3)
if len(cur):
    last_doy = cur.index[-1]
    diff = cur.iloc[-1] - band_mean.loc[last_doy]
    ax.annotate(f"{this_year}: {cur.iloc[-1]:.0f} mm\n({diff:+.0f} mm vs. average)",
                xy=(to_dates(pd.Index([last_doy]))[0], cur.iloc[-1]), xytext=(8, -6),
                textcoords="offset points", ha="left", va="top", color=INK, fontsize=10)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set_ylabel("Cumulative rain since 1 Jan (mm)")
ax.set_title(f"Münster rainfall {this_year} vs. {past.columns.min()}–{past.columns.max()}")
ax.legend(frameon=False, loc="upper left")
fig.text(0.01, 0.01, SOURCE_NOTE, color=MUTED, fontsize=8)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(FIG / "02_cumulative_year.png")
plt.show()

## 5. Chart 3 - summer rain and longest dry spell by year

In [ ]:
def longest_dry_spell(s: pd.Series, thr: float = DRY_DAY_MM):
    """Longest run of consecutive dry days. Missing days break a run (conservative)."""
    best, run, best_end = 0, 0, None
    for d, v in s.items():
        run = run + 1 if (pd.notna(v) and v < thr) else 0
        if run > best:
            best, best_end = run, d
    return best, best_end

rows = []
for y in range(BASELINE_START, SUMMER_YEAR + 1):
    s = daily[(daily.index.year == y) & (daily.index.month.isin(SUMMER_MONTHS))]
    if s.notna().mean() < 0.9:
        continue
    spell, spell_end = longest_dry_spell(s)
    rows.append({"year": y, "summer_mm": s.sum(), "dry_days": int((s < DRY_DAY_MM).sum()),
                 "longest_dry_spell": spell, "spell_end": spell_end})
summer = pd.DataFrame(rows).set_index("year")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
colors_rain = [BLUE if y == SUMMER_YEAR else BLUE_LIGHT for y in summer.index]
colors_dry = [ORANGE if y == SUMMER_YEAR else GRAY_BAR for y in summer.index]

ax = axes[0]
ax.bar(summer.index, summer["summer_mm"], color=colors_rain, width=0.8, zorder=2)
ax.axhline(summer["summer_mm"].mean(), color=MUTED, ls="--", lw=1.2)
ax.text(summer.index.min() - 0.4, summer["summer_mm"].mean(), " avg", va="bottom", color=MUTED, fontsize=9)
ax.set_title("Summer rain (Jun–Aug, mm)")

ax = axes[1]
ax.bar(summer.index, summer["longest_dry_spell"], color=colors_dry, width=0.8, zorder=2)
ax.set_title("Longest dry spell in summer (days)")

for ax, col in zip(axes, ["summer_mm", "longest_dry_spell"]):
    if SUMMER_YEAR in summer.index:
        v = summer.loc[SUMMER_YEAR, col]
        ax.text(SUMMER_YEAR, v, f"{v:.0f}", ha="center", va="bottom", color=INK, fontsize=10, fontweight="bold")
    ax.set_xticks(summer.index[::2])
fig.suptitle(f"Summer {SUMMER_YEAR} in Münster, against every summer since {summer.index.min()}",
             x=0.01, ha="left", fontweight="bold", fontsize=14, color=INK)
fig.text(0.01, 0.01, f"Dry day = under {DRY_DAY_MM:g} mm. " + SOURCE_NOTE, color=MUTED, fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 0.97))
fig.savefig(FIG / "03_summer_drought.png")
plt.show()
summer.round({"summer_mm": 1})

## 6. Headline numbers + draft post

Check every number against the charts before posting. Edit the text so it sounds like you.

In [ ]:
s_row = summer.loc[SUMMER_YEAR] if SUMMER_YEAR in summer.index else None
wettest_day = week.idxmax()

print("=== Headline numbers ===")
print(f"Last 7 days ({week_start:%d %b}–{week_end:%d %b}): {week_total:.1f} mm "
      f"(same week avg {avg_week:.1f} mm, x{week_total / avg_week:.1f}; rank #{rank} of {len(same_week) + 1})")
print(f"Wettest day: {wettest_day:%A %d %b} with {week.max():.1f} mm")
if s_row is not None:
    print(f"Summer {SUMMER_YEAR}: {s_row.summer_mm:.0f} mm (avg {summer.summer_mm.mean():.0f} mm), "
          f"{s_row.dry_days} dry days, longest dry spell {s_row.longest_dry_spell} days "
          f"(ending {s_row.spell_end:%d %b}); rank by dry spell: "
          f"#{int((summer.longest_dry_spell > s_row.longest_dry_spell).sum()) + 1} of {len(summer)}")

post = f"""
This week Münster got {week_total:.0f} mm of rain in 7 days. The same week has averaged {avg_week:.0f} mm since {same_week.index.min()}.
"""
if s_row is not None:
    post += f"""
Just a few months ago it was the opposite: summer {SUMMER_YEAR} brought {s_row.summer_mm:.0f} mm (average: {summer.summer_mm.mean():.0f} mm), \
with {s_row.longest_dry_spell} days in a row without meaningful rain.
"""
post += """
Averages hide extremes. I'm building an open, end-to-end tracker for rain and drought extremes in Münster, \
using DWD open data: radar, forecasts and soil moisture, then a risk model and a live map with plain-language alerts.

I'll share the build here as I go: data, code, mistakes and all. Code: <your GitHub link>

What would you want a local tool like this to tell you?

#Münster #Klimawandel #OpenData #DataScience #ClimateTech
"""
print("\n=== Draft post ===")
print(post)